In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## ជំហ៊ានទី ១: កំណត់ម៉ូឌែល Pydantic សម្រាប់លទ្ធផលមានរចនាសម្ព័ន្ធ

ម៉ូឌែលទាំងនេះកំណត់ **គំរូ** ដែលភ្នាក់ងារនឹងផ្តល់វិញ។ ការប្រើ `response_format` ជាមួយ Pydantic នឹងធ្វើឱ្យមាន៖
- ✅ ការដកព័ត៌មានប្រភេទសុវត្ថិភាព
- ✅ ការត្រួតពិនិត្យស្វ័យប្រវត្តិ
- ✅ គ្មានកំហុសបំលែងពីចម្លើយអត្ថបទសេរី
- ✅ ការបញ្ជូនលក្ខខណ្ឌបានងាយស្រួលដោយផ្អែកលើវាលផ្សេងៗ


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ជំហានទី 2: បង្កើតឧបករណ៍កក់សណ្ឋាគារ

ឧបករណ៍នេះគឺជាអ្វីដែល **availability_agent** នឹងហៅដើម្បីពិនិត្យមើលថាតើបន្ទប់មានអាចប្រើបានឬទេ។ យើងប្រើ `@ai_function` decorator ដើម្បី៖
- បំលែងមុខងារ Python មួយទៅជាឧបករណ៍អាចហៅដោយ AI
- បង្កើត schemas JSON សម្រាប់ LLM ដោយស្វ័យប្រវត្តិ
- ប្រតិបត្តិនូវការត្រួតពិនិត្យប៉ារ៉ាម៉ែត្រ
- អនុញ្ញាតិឱ្យត្រូវហៅដោយស្វ័យប្រវត្តិដោយភ្នាក់ងារ

សម្រាប់ការបង្ហាញនេះ៖
- **Stockholm, Seattle, Tokyo, London, Amsterdam** → មានបន្ទប់ ✅
- **ទីក្រុងផ្សេងទៀតទាំងអស់** → មិនមានបន្ទប់ ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ជំហានទី ៣៖ កំណត់ פונקציותលក្ខខណ្ឌសម្រាប់ការបង្ហាញផ្លូវ

פונקציותទាំងនេះពិនិត្យនូវការឆ្លើយតបរបស់ភ្នាក់ងារ ហើយកំណត់ថាត្រូវទៅផ្លូវណា ក្នុងដំណើរការងារ។

**លំនាំសំខាន់៖**
១. ពិនិត្យមើលថាសារមាន `AgentExecutorResponse` ឬទេ
២. បកស្រាយអ្វីដែលបានចេញ (ម៉ូដែល Pydantic)
៣. ត្រឡប់ `True` ឬ `False` ដើម្បីបង្រួចផ្លូវ

ដំណើរការងារនឹងវាយតម្លៃលក្ខខណ្ឌទាំងនេះលើ **គម្លាត** ដើម្បីសម្រេចថាតើត្រូវហៅកម្មវិធីចេញណាមួយបន្ទាប់។


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## ជំហ៊ាន 4៖ បង្កើតកម្មញ្ជកថា Custom Display Executor

កម្រើកថាអ្នកបញ្ចប់ម្នាក់គឺជាផ្នែករបស់ការងារផ្លូវការដែលបំលែង ឬបង្កើតផលប៉ះពាល់។ យើងប្រើ `@executor` ដើម្បីបង្កើតកម្មញ្ជកថាប្រភេទផ្ទាល់ខ្លួនដែលបង្ហាញលទ្ធផលចុងក្រោយ។

**គំនិតសំខាន់ៗ៖**
- `@executor(id="...")` - ចុះបញ្ជីមុខងារជា executor នៃ workflow
- `WorkflowContext[Never, str]` - ពន្លកប្រភេទសម្រាប់បញ្ចូល/បញ្ចីចេញ
- `ctx.yield_output(...)` - ផ្តល់លទ្ធផលចុងក្រោយនៃ workflow


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## ជំហានទី ៥៖ ផ្ទុកអថេរសេវាកម្មបរិយាកាស

កំណត់រចនាសម្ព័ន្ធអ្នកអតិថិជន LLM។ ឧទាហរណ៍នេះដំណើរការជាមួយ៖
- **ម៉ូដែល GitHub** (ជាន់ដើមរថតនៅជាមួយសញ្ញាអត្តសញ្ញាណ GitHub)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ជំហានទី ៦៖ បង្កើតតំណាង AI ជាមួយលទ្ធផលដែលមានរចនាសម្ព័ន្ធ

យើងបង្កើត **តំណាងពិសេសបីនាក់** ម្នាក់ៗត្រូវបានបញ្ចូលនៅក្នុង `AgentExecutor`៖

១. **availability_agent** - ពិនិត្យភាពមានសម្បូរបែបនៃសណ្ឋាគារដោយប្រើឧបករណ៍
២. **alternative_agent** - ស្នើជាទីក្រុងជំនួស (ពេលគ្មានបន្ទប់)
៣. **booking_agent** - លើកទឹកចិត្តកិច្ចការកក់ (ពេលមានបន្ទប់)

**លក្ខណៈសំខាន់៖**
- `tools=[hotel_booking]` - ផ្តល់ឧបករណ៍ទៅឲ្យតំណាង
- `response_format=PydanticModel` - បង្ខំឲ្យមានលទ្ធផល JSON ដែលមានរចនាសម្ព័ន្ធ
- `AgentExecutor(..., id="...")` - បញ្ចូលតំណាងសម្រាប់ប្រើប្រាស់ក្នុងសកម្មភាព


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## ជំហានទី 7៖ សាងសង់ Workflow ជាមួយនឹងគោលល្នឹមមានលក្ខខណ្ឌ

ឥឡូវនេះយើងប្រើប្រាស់ `WorkflowBuilder` ដើម្បីសាងសង់ក្រាបជាមួយនឹងផ្លូវរាំងកិច្ចតាមលក្ខខណ្ឌ៖

**រចនាសម្ព័ន្ធ Workflow៖**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**វិធីសាស្រ្តសំខាន់ៗ៖**
- `.set_start_executor(...)` - កំណត់ចំណុចចូល
- `.add_edge(from, to, condition=...)` - បន្ថែមលំនឹងមានលក្ខខណ្ឌ
- `.build()` - បញ្ចប់ workflow


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## ជំហានទី 8: ប្រតិបត្តិការ​ករណី​សាកល្បង​លេខ 1 - ទីក្រុងដែលមិនមានការចូលប្រើ (ប៉ារីស)

យើង​ត្រូវសាកល្បងផ្លូវ **មិនមានការចូលប្រើ** ដោយស្នើសុំសណ្ឋាគារនៅប៉ារីស (ដែលមិនមានបន្ទប់ក្នុងការសមមាត្រ​របស់​យើង)។


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Step 9: បើកដំណើរការ Test Case 2 - ទីក្រុងមាន Availability (Stockholm)

ឥឡូវនេះចង់សាកល្បងផ្លូវ **availability** ដោយស្នើសុំសណ្ឋាគារនៅទីក្រុង Stockholm (ដែលមានបន្ទប់នៅក្នុងការសម្រាប់សាកល្បងរបស់យើង)។


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## ចំណុចសំខាន់ៗ និងជំហានទៅមុខ

### ✅ អ្វីដែលអ្នកបានរៀន៖

1. **ចំណង់ WorkflowBuilder**
   - ប្រើ `.set_start_executor()` ដើម្បីកំណត់ចំណុចចូល
   - ប្រើ `.add_edge(from, to, condition=...)` សម្រាប់ការផ្លូវលំស៊ីត
   - ហៅ `.build()` ដើម្បីបញ្ចប់ workflow

2. **ផ្លូវលំស៊ីត**
   - មុខងារតម្រៀបពិនិត្យ `AgentExecutorResponse`
   - វិភាគលទ្ធផលរៀបចំរចនាដើម្បីសម្រេចផ្លូវ
   - បង្វិល `True` ដើម្បីបើកផ្លូវ, `False` ដើម្បីរំលងវា

3. **ការរួមបញ្ចូលឧបករណ៍**
   - ប្រើ `@ai_function` ដើម្បីបម្លែងមុខងារ Python ទៅជាឧបករណ៍ AI
   - អ្នកតំណាងហៅឧបករណ៍ដោយស្វ័យប្រវត្តិពេលត្រូវការ
   - ឧបករណ៍ត្រឡប់ JSON ដែលអ្នកតំណាងអាចវិភាគបាន

4. **លទ្ធផលរៀបចំរចនា**
   - ប្រើគំរូ Pydantic សម្រាប់ការដកទិន្នន័យប្រភេទសុវត្ថិភាព
   - កំណត់ `response_format=MyModel` ពេលបង្កើតអ្នកតំណាង
   - វិភាគចម្លើយជាមួយ `Model.model_validate_json()`

5. **អ្នកបម្រែបម្រួលផ្ទាល់ខ្លួន**
   - ប្រើ `@executor(id="...")` ដើម្បីបង្កើតធាតុ workflow
   - អ្នកបម្រែបម្រួលអាចបំលែងទិន្នន័យឬអនុវត្តផលប៉ៈពាល់ក្រោយ
   - ប្រើ `ctx.yield_output()` ដើម្បីផលិតលទ្ធផល workflow

### 🚀 ការអនុវត្តនៅពិភពជាក់ស្តែង៖

- **ការកក់ដំណើរកំសាន្ត**៖ ពិនិត្យavailability, ផ្ដល់ជម្រើសជំនួស, ប្រៀបធៀបជម្រើស
- **សេវាកម្មអតិថិជន**៖ ផ្លូវលំស៊ីតដោយយោងទៅលើប្រភេទបញ្ហា, អារម្មណ៍, អាទិភាព
- **កម្មវិធីអេឡិចត្រូនិក**៖ ពិនិត្យស្តុក, ផ្ដល់ជម្រើសជំនួស, ដំណើរការបញ្ជាទិញ
- **ការត្រួតពិនិត្យមាតិកា**៖ ផ្លូវលំស៊ីតដោយគិតពីពិន្ទុពុល, ទង់ស្លាកអ្នកប្រើ
- **Workflow អនុម័ត**៖ ផ្លូវលំស៊ីតដោយយោងទៅលើចំនួន, តួនាទីអ្នកប្រើ, កម្រិតហានិភ័យ
- **ការដំណើរការជាបន្ទាប់**៖ ផ្លូវលំស៊ីតដោយគុណភាពទិន្នន័យ, សម្រួលភាព

### 📚 ជំហានបន្ទាប់៖

- បន្ថែមលក្ខខណ្ឌស្មុគស្មាញបន្ថែម (លក្ខខណ្ឌច្រើន)
- អនុវត្តរង្វិលជាមួយការគ្រប់គ្រងស្ថានភាព workflow
- បន្ថែម sub-workflow សម្រាប់ធាតុអាចប្រើឡើងវិញ
- រួមបញ្ចូលជាមួយ API ជាក់ស្តែង (កក់សណ្ឋាគារ, ប្រព័ន្ធស្តុក)
- បន្ថែមការដោះស្រាយកំហុស និងផ្លូវជំនួស
- បង្ហាញ workflow ដោយប្រើឧបករណ៍បង្ហាញ built-in


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
